# 自動化 Face LoRA 生產線 (Google Colab 執行版)

這個筆記本將幫助您在 Google Colab 上自動執行 Face LoRA 生產線。
請確保您已經將整個專案資料夾上傳到了 Google Drive 的 `MyDrive/Face_LoRA_Pipeline` 目錄下。

## Training


In [ ]:
# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 切換到專案工作目錄
import os

# PROJECT_DIR = '/content/drive/MyDrive/Face_LoRA_Pipeline'
PROJECT_DIR = '/content/drive/MyDrive/app/AI/Lora/Face_LoRA_Pipeline'
if not os.path.exists(PROJECT_DIR):
    print(f"錯誤：找不到路徑 {PROJECT_DIR}")
    print("請確保您已將專案上傳至 Google Drive 的正確位置。")
else:
    os.chdir(PROJECT_DIR)
    print(f"成功切換至工作目錄: {os.getcwd()}")

In [ ]:
# 3. 下載並安裝 kohya_ss 訓練環境
# 將 kohya_ss 腳本直接下載到 Colab 虛擬機本機端，以避免 Google Drive 斷線問題
import os
KOHYA_DIR = "/content/kohya_ss"

if not os.path.exists(KOHYA_DIR):
    print("尚未下載 kohya_ss，開始從 GitHub 複製...")
    !git clone --recursive https://github.com/bmaltais/kohya_ss.git {KOHYA_DIR}
else:
    print("已在 Google Drive 中找到 kohya_ss 腳本，跳過下載步驟。")

# 切換至 kohya_ss 目錄
%cd {KOHYA_DIR}

# 確保子模組 (如 sd-scripts) 已正確初始化並更新
!git submodule update --init --recursive

# 【重要說明】：即使腳本存在 Drive 中，但 Colab 每次啟動都是一台全新的虛擬機。
# 因此 Python 套件必須「每次」重新安裝，否則會報錯找不到套件。
!pip install -r requirements.txt
!pip install accelerate transformers diffusers

# 安裝專案本身的依賴套件 (OpenCV, InsightFace 等)
!pip install opencv-python-headless insightface

# 切換回專案目錄
%cd {PROJECT_DIR}

In [ ]:
# 4. 執行全自動生產線主程式
# (請確認已經修改了 src/training/kohya_runner.py 移除了 Mock)
!python main.py

## 5. 獨立評分測試 

In [ ]:
#@title 5. 獨立評分測試 (Standalone Evaluation Testing)
# 這個區塊完全獨立於上面的流程，只要您的模型已經訓練好，就可以直接單獨執行這裡來產圖並查看評分。

# 5.1 安裝必要的套件 (獨立安裝確保環境乾淨)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q diffusers==0.27.2 peft==0.10.0 transformers==4.40.0 accelerate==0.30.0 insightface onnxruntime-gpu huggingface-hub==0.25.2

import os
import torch
import cv2
import numpy as np
import insightface
from diffusers import StableDiffusionPipeline
from google.colab.patches import cv2_imshow

# 5.2 定義變數與路徑
FACE_ID = "Tzuyu"  # ⚠️請替換為您剛剛訓練的臉部名稱
PROJECT_DIR = "/content/drive/MyDrive/app/AI/Lora/Face_LoRA_Pipeline"
MODEL_PATH = f"{PROJECT_DIR}/output/models/{FACE_ID}.safetensors"
BASE_MODEL_ID = "runwayml/stable-diffusion-v1-5" # 基底模型
ORIGINAL_IMAGES_DIR = f"{PROJECT_DIR}/training_data/{FACE_ID}"
EVAL_OUTPUT_DIR = f"{PROJECT_DIR}/output/eval/{FACE_ID}_standalone"

PROMPT = f"A portrait of {FACE_ID}, raw photo, highly detailed, 8k uhd, dslr"
NEGATIVE_PROMPT = "blurry, out of focus, disfigured, low quality, bad anatomy"
NUM_IMAGES = 10

os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

# 5.3 載入產圖模型 (Diffusers)
print("載入產圖模型中...")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"找不到您的 LoRA 模型：{MODEL_PATH}，請確認檔名與路徑是否正確！")

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=dtype, safety_checker=None).to(device)
lora_dir = os.path.dirname(MODEL_PATH)
lora_file = os.path.basename(MODEL_PATH)
pipe.load_lora_weights(lora_dir, weight_name=lora_file)

# 5.4 開始產圖
generated_paths = []
print(f"開始產生 {NUM_IMAGES} 張測試圖片...")
for i in range(NUM_IMAGES):
    out_path = os.path.join(EVAL_OUTPUT_DIR, f"{FACE_ID}_standalone_{i}.jpg")
    img = pipe(prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT, num_inference_steps=30, guidance_scale=7.5).images[0]
    img.save(out_path, format="JPEG", quality=95)
    generated_paths.append(out_path)
    print(f"已儲存: {out_path}")

del pipe
torch.cuda.empty_cache()

# 5.5 載入評分模型 (InsightFace)
print("\n載入 InsightFace 評分模型中...")
app = insightface.app.FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

def get_embedding(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    faces = app.get(img)
    return faces[0].normed_embedding if faces else None

# 5.6 計算相似度
print("\n開始計算相似度...")
original_images = [os.path.join(ORIGINAL_IMAGES_DIR, f) for f in os.listdir(ORIGINAL_IMAGES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

total_score = 0
comparisons = 0

for gen_path in generated_paths:
    gen_emb = get_embedding(gen_path)
    if gen_emb is None:
        print(f"\n--- ⚠️ 產出圖片 {os.path.basename(gen_path)}: 未偵測到人臉或圖片損壞，跳過評估 ---")
        continue
    
    print(f"\n--- 評估產出圖片: {os.path.basename(gen_path)} ---")
    display_img = cv2.imread(gen_path)
    display_img = cv2.resize(display_img, (256, 256))
    cv2_imshow(display_img)
    
    for orig_path in original_images:
        orig_emb = get_embedding(orig_path)
        if orig_emb is None:
            print(f"⚠️ 警告：無法在原圖 (檔名: {os.path.basename(orig_path)}) 中偵測到人臉，已跳過該 Ground Truth 圖片的比對。")
            continue
        
        score = np.dot(gen_emb, orig_emb)
        total_score += score
        comparisons += 1

if comparisons > 0:
    avg_score = (total_score / comparisons) * 100
    print(f"\n======================================")
    print(f"最終平均相似度評分: {avg_score:.2f}%")
    print(f"======================================")
else:
    print("無法計算相似度，可能是圖片中未偵測到人臉。")










## 6. 升級版：自動化驗收與黃金引數決策系統 (Standalone Pipeline)
基於 PRD 實作的兩階段驗收管線（文生圖 + Inpainting），將結果寫入註冊表。

In [ ]:
#@title 6. 自動化驗收與黃金引數決策系統 (兩階段驗收管線)
# 此區塊會自動執行文生圖天花板測試與 Inpainting 壓力測試，並更新 lora_registry.json
import os
import sys
import json
import uuid
import torch
import cv2
import numpy as np
import pandas as pd
from PIL import Image

# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. 匯入全域 config.py (唯讀 config.py 鐵律)
PROJECT_DIR = "/content/drive/MyDrive/app/AI/Lora/Face_LoRA_Pipeline"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
import config

FACE_ID = "Tzuyu"  # ⚠️請替換為您剛剛訓練的臉部名稱
MODEL_PATH = os.path.join(config.MODELS_DIR, f"{FACE_ID}.safetensors")
BENCHMARK_IMAGES_DIR = os.path.join(config.BASE_DIR, "benchmark_images")
ORIGINAL_IMAGES_DIR = os.path.join(config.OUTPUT_DIR, "preprocessed_data", FACE_ID, f"10_{FACE_ID}")
EVAL_OUTPUT_DIR = config.EVAL_OUTPUT_DIR_TEMPLATE.format(face_id=FACE_ID)
REGISTRY_FILE = config.REGISTRY_FILE

print(f"=== 🛫 起飛前環境與檔案檢查 ===")
print(f"1. 測試對象 (FACE_ID): {FACE_ID}")
if not os.path.exists(MODEL_PATH):
    print(f"❌ 錯誤: 找不到訓練好的模型 {MODEL_PATH}")
    sys.exit("檢查失敗，終止執行。")
else:
    print(f"✅ 找到模型: {MODEL_PATH}")

print("2. 檢查 benchmark_images 靶圖...")
if os.path.exists(BENCHMARK_IMAGES_DIR):
    files = [f for f in os.listdir(BENCHMARK_IMAGES_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]
    scenes = ['standard', 'close_up', 'wide', 'low_light', 'profile']
    matched = sum(1 for s in scenes if any(s.replace("_", "") in os.path.splitext(f)[0].lower().replace(" ", "").replace("-", "").replace("_", "") for f in files))
    print(f"   成功辨識出 {matched}/5 個場景的標準靶圖。")
else:
    print(f"⚠️ 警告: 找不到 {BENCHMARK_IMAGES_DIR}，系統將自動略過 Inpainting 壓力測試。")

print("3. Ground Truth 入站安檢 (Fail-Fast Validation)...")
if not os.path.exists(ORIGINAL_IMAGES_DIR):
    print(f"❌ 錯誤: 找不到 Ground Truth 資料夾 {ORIGINAL_IMAGES_DIR}")
    sys.exit("檢查失敗，終止執行。")

gt_files = [f for f in os.listdir(ORIGINAL_IMAGES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
if not gt_files:
    print(f"❌ 錯誤: Ground Truth 資料夾內無圖片。")
    sys.exit("檢查失敗，終止執行。")

# 解析度攔截
for f in gt_files:
    img_path = os.path.join(ORIGINAL_IMAGES_DIR, f)
    img = Image.open(img_path)
    if img.size != (config.TARGET_FACE_RESOLUTION, config.TARGET_FACE_RESOLUTION):
        print(f"❌ 致命錯誤: Ground Truth 圖片 {f} 解析度為 {img.size}，不符合 Config 規定的 {config.TARGET_FACE_RESOLUTION}x{config.TARGET_FACE_RESOLUTION}。")
        print("拒絕進行不對等的尺規污染測試，請重新執行預處理！")
        sys.exit("解析度安檢失敗，終止執行。")
print(f"✅ Ground Truth 安檢通過，共 {len(gt_files)} 張，解析度皆為 {config.TARGET_FACE_RESOLUTION}x{config.TARGET_FACE_RESOLUTION}。")

print("====================================")
confirm = input("⚠️ 請確認以上變數與檔案皆已到位。\n(按 Enter 繼續執行，或輸入 'q' 取消): ")
if confirm.lower() == 'q':
    sys.exit("使用者取消執行。")
print("檢查通過，開始安裝相依套件與載入模型 (需要幾分鐘，請稍候)...\n")

# 6.1 安裝黃金相容性套件
!pip install -q diffusers==0.27.2 peft==0.10.0 transformers==4.40.0 accelerate==0.30.0 insightface onnxruntime-gpu huggingface-hub==0.25.2

from diffusers import AutoPipelineForText2Image, AutoPipelineForInpainting
import insightface

os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 水印函式
def add_watermark_cv2(img_cv2, text):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    font_thickness = 2
    text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
    margin = 20
    x = img_cv2.shape[1] - text_size[0] - margin
    y = img_cv2.shape[0] - margin
    overlay = img_cv2.copy()
    cv2.rectangle(overlay, (x - 10, y - text_size[1] - 10), (x + text_size[0] + 10, y + 10), (0, 0, 0), -1)
    alpha = 0.5
    cv2.addWeighted(overlay, alpha, img_cv2, 1 - alpha, 0, img_cv2)
    cv2.putText(img_cv2, text, (x, y), font, font_scale, (255, 255, 255), font_thickness, cv2.LINE_AA)
    return img_cv2

# ==========================================
# 第一階段 (產圖期)：模型解耦與記憶體釋放
# ==========================================
print("\n=== [第一階段] 產圖期：文生圖天花板測試 (Text-to-Image Baseline) ===")
pipe_t2i = AutoPipelineForText2Image.from_pretrained(config.EVAL_MODEL_CHECKPOINT, torch_dtype=dtype, safety_checker=None).to(device)
lora_dir = os.path.dirname(MODEL_PATH)
lora_file = os.path.basename(MODEL_PATH)
pipe_t2i.load_lora_weights(lora_dir, weight_name=lora_file)

lora_scales_t2i = [0.6, 0.8, 1.0, 1.2]
t2i_records = []

for scene, prompt in config.EVAL_PROMPTS.items():
    prompt = prompt.format(trigger_word=FACE_ID)
    for scale in lora_scales_t2i:
        generator = torch.Generator(device=device).manual_seed(config.FIXED_SEED)
        identifier = f"T2I_{uuid.uuid4().hex[:8].upper()}"
        out_path = os.path.join(EVAL_OUTPUT_DIR, f"{identifier}.jpg")
        
        img = pipe_t2i(prompt=prompt, negative_prompt=config.EVAL_NEGATIVE_PROMPT, num_inference_steps=30, guidance_scale=7.5, cross_attention_kwargs={"scale": scale}, generator=generator).images[0]
        img_cv2 = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        img_cv2 = add_watermark_cv2(img_cv2, identifier)
        cv2.imwrite(out_path, img_cv2)
        
        t2i_records.append({"identifier": identifier, "scene": scene, "prompt": prompt, "lora_scale": scale, "denoising": None, "path": out_path})
        print(f"[{scene}] Scale: {scale} -> 產出完成: {identifier}")

del pipe_t2i
torch.cuda.empty_cache()

print("\n=== [第一階段] 產圖期：Inpainting 壓力測試 (Inpainting Matrix) ===")
inpaint_records = []
if not os.path.exists(BENCHMARK_IMAGES_DIR) or len(os.listdir(BENCHMARK_IMAGES_DIR)) == 0:
    print(f"⚠️ 警告：找不到 {BENCHMARK_IMAGES_DIR} 目錄或目錄為空，跳過 Inpainting 測試。")
else:
    pipe_inp = AutoPipelineForInpainting.from_pretrained(config.EVAL_MODEL_CHECKPOINT, torch_dtype=dtype, safety_checker=None).to(device)
    pipe_inp.load_lora_weights(lora_dir, weight_name=lora_file)
    
    lora_scales_inp = [0.5, 0.7, 0.9, 1.1]
    denoising_strengths = [0.35, 0.50, 0.65, 0.80]
    mask_image = Image.new("RGB", (1024, 1024), (255, 255, 255))
    
    for scene, prompt in config.EVAL_PROMPTS.items():
        prompt = prompt.format(trigger_word=FACE_ID)
        search_term = scene.replace("scene_", "").replace("_", "").lower()
        target_img_files = [f for f in os.listdir(BENCHMARK_IMAGES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg")) and search_term in os.path.splitext(f)[0].lower().replace(" ", "").replace("-", "").replace("_", "")]
        if not target_img_files:
            print(f"⚠️ 警告：找不到場景 '{scene}' 的標靶圖，跳過此場景所有 Inpainting 測試 (無產出圖檔代號)！")
            continue
        
        target_img_path = os.path.join(BENCHMARK_IMAGES_DIR, target_img_files[0])
        init_image = Image.open(target_img_path).convert("RGB").resize((1024, 1024))
        
        for scale in lora_scales_inp:
            for denoise in denoising_strengths:
                generator = torch.Generator(device=device).manual_seed(config.FIXED_SEED)
                identifier = f"INP_{uuid.uuid4().hex[:8].upper()}"
                out_path = os.path.join(EVAL_OUTPUT_DIR, f"{identifier}.jpg")
                
                img = pipe_inp(prompt=prompt, negative_prompt=config.EVAL_NEGATIVE_PROMPT, image=init_image, mask_image=mask_image, num_inference_steps=30, strength=denoise, guidance_scale=7.5, cross_attention_kwargs={"scale": scale}, generator=generator).images[0]
                img_cv2 = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
                img_cv2 = add_watermark_cv2(img_cv2, identifier)
                cv2.imwrite(out_path, img_cv2)
                
                inpaint_records.append({"identifier": identifier, "scene": scene, "prompt": prompt, "lora_scale": scale, "denoising": denoise, "path": out_path})
                print(f"[{scene}] Scale: {scale}, Denoise: {denoise} -> 產出完成: {identifier}")

    del pipe_inp
    torch.cuda.empty_cache()

# ==========================================
# 第二階段 (評估期)：確立自體基準線與多尺度重試
# ==========================================
print("\n=== [第二階段] 評估期：載入 InsightFace 評分模型 ===")
# 移除 det_size=(640, 640) 解除尺寸封印
app = insightface.app.FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0)

def extract_embedding(img_cv2):
    faces = app.get(img_cv2)
    return faces[0].normed_embedding if faces else None

def get_embedding_with_fallback(img_path, use_fallback=False):
    img = cv2.imread(img_path)
    if img is None: return None
    emb = extract_embedding(img)
    if emb is not None: return emb
    
    if not use_fallback: return None
    
    # 動態多尺度重試機制
    print(f"  ⚠️ 警告：原尺寸偵測失敗，啟用多尺度備案偵測...")
    h, w = img.shape[:2]
    img_200 = cv2.resize(img, (w*2, h*2))
    emb_200 = extract_embedding(img_200)
    if emb_200 is not None:
        print(f"  ✅ 備案成功：透過 200% 放大成功擷取特徵。")
        return emb_200
        
    img_50 = cv2.resize(img, (w//2, h//2))
    emb_50 = extract_embedding(img_50)
    if emb_50 is not None:
        print(f"  ✅ 備案成功：透過 50% 縮小成功擷取特徵。")
        return emb_50
        
    return None

print("\n=== 確立自體相似度天花板 (Intra-Class Baseline) ===")
gt_embeddings = []
for f in gt_files:
    img_path = os.path.join(ORIGINAL_IMAGES_DIR, f)
    emb = get_embedding_with_fallback(img_path, use_fallback=False) # GT 不應需要 fallback，因為已過質檢
    if emb is not None:
        gt_embeddings.append(emb)
    else:
        print(f"⚠️ 警告：無法在 Ground Truth 原圖 (檔名: {f}) 中偵測到人臉，已跳過該圖。")

if len(gt_embeddings) < 2:
    print("⚠️ 警告：成功擷取特徵的 Ground Truth 數量不足 2 張，無法建立自體基準線。")
    print("⚠️ 警告：已自動降級為絕對計分模式 (基準線設為 100%)，繼續執行。")
    baseline_score = 1.0
else:

    cross_scores = []
    for i in range(len(gt_embeddings)):
        for j in range(i + 1, len(gt_embeddings)):
            cross_scores.append(np.dot(gt_embeddings[i], gt_embeddings[j]))

    baseline_score = float(np.mean(cross_scores))
    print(f"✅ [動態基準] 交叉比對基準線 (滿分天花板) 為: {baseline_score * 100:.2f}%")

print("\n=== 開始計算標準化分數 ===")
def evaluate_records(records):
    for r in records:
        emb = get_embedding_with_fallback(r['path'], use_fallback=True)
        if emb is None:
            print(f"⚠️ 警告：產出圖片代號 [{r['identifier']}] (檔名: {os.path.basename(r['path'])}) 三維度偵測皆無人臉，給予 0 分。")
            r['score'] = 0.0
            continue
            
        scores = [np.dot(emb, gt) for gt in gt_embeddings]
        abs_score = float(np.mean(scores))
        normalized_score = min(1.0, abs_score / baseline_score) * 100
        r['score'] = normalized_score
        print(f"[{r['identifier']}] 原始分: {abs_score*100:.2f}% / 基準 {baseline_score*100:.2f}% -> 標準化得分: {normalized_score:.2f}%")
        
evaluate_records(t2i_records)
evaluate_records(inpaint_records)

# ==========================================
# 儲存日誌與更新註冊表
# ==========================================
print("\n=== 彙整黃金引數與更新註冊表 ===")
df = pd.DataFrame(t2i_records + inpaint_records)
df = df.drop(columns=['path']) # 不需匯出本地路徑
df.to_csv(os.path.join(EVAL_OUTPUT_DIR, "evaluation_matrix.csv"), index=False)

golden_rules = {}
for scene in config.EVAL_PROMPTS.keys():
    scene_df = df[df['scene'] == scene]
    if not scene_df.empty:
        best_row = scene_df.loc[scene_df['score'].idxmax()]
        golden_rules[scene] = {
            "best_identifier": best_row['identifier'],
            "lora_scale": float(best_row['lora_scale']),
            "denoising": float(best_row.get('denoising', 0.50)) if 'denoising' in best_row and not pd.isna(best_row.get('denoising')) else 0.50,
            "max_score": float(best_row['score'])
        }

if os.path.exists(REGISTRY_FILE):
    with open(REGISTRY_FILE, 'r', encoding='utf-8') as f:
        registry = json.load(f)
    
    for face in registry.get('faces', []):
        if face.get('face_id') == FACE_ID:
            face['golden_rules'] = golden_rules
            break
            
    with open(REGISTRY_FILE, 'w', encoding='utf-8') as f:
        json.dump(registry, f, indent=4, ensure_ascii=False)
    print(f"✅ 已成功將黃金引數與最佳代號寫入 {REGISTRY_FILE} !")
else:
    print(f"⚠️ 警告：找不到註冊表 {REGISTRY_FILE}，無法寫入。")

